## Evaluation for Disambiguation
The evaluation for disambiguation will be split up into three stages:
1) Parsing the evaluation set
2) Creating the gold standard from the eval set (done by hand)
3) Evaluating the model on the gold standard

### Parsing the evaluation set
For our evaluation set, we will be using a sample of 4646 example sentences from the OPD, of which we will sample sentences that are fully analyzed by the FST (but could contain no ambiguities). The code used to sample the tsv file and create the evaluation set lives in the `eval_modules/` folder.

**Note:** It's best to skip re-running the following cell, as this will write over the manually corrected files (as is detailed in the following section). If needed, uncomment and run. 

In [2]:
import os
os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

In [ ]:
from grammar_modules.disambiguation import REPO_ROOT
from eval_modules.data_io import read_opd_tsv, build_eval_sample, write_eval_artifacts

TSV_PATH = REPO_ROOT / "data" / "opd" / "example_sentences.tsv"
FST_BINARY_PATH = REPO_ROOT / "data" / "fst" / "ojibwe.fomabin"
OUTDIR = REPO_ROOT / "evaluation" / "eval_data" / "sample_disambig"
filename = "sample_500"

# read tsv
rows = read_opd_tsv(TSV_PATH)
# build the evaluation set with random seed
keep = build_eval_sample(rows, FST_BINARY_PATH, sample_size=500, seed=421)
# write the evaluation sample sets
write_eval_artifacts(keep, FST_BINARY_PATH, OUTDIR, filename, write_100=False)

print("Wrote files:",
      OUTDIR / "sample_500.tsv",
      OUTDIR / "sample_500.txt",
      sep="\n")


FileNotFoundError: /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin

### Creating the gold standard

The gold standard was created by doing disambiguation by hand on the sample sets parsed above. Incorrect lines were deleted from the `.cg3` file by hand, as if the human was the disambiguation module. In total, 492 sentence examples were disambiguated for the gold standard, which is located in The gold set is in `eval_data/gold/disambig_gold_492.txt`.

Originally, due to a misparse sentence #30 was replaced with #300, which was then replaced with #500.

On a second look, more sentences than anticipated contained typos or FST misparses. Therefore, the following sentence were manually removed from the gold standard: 124, 201, 239, 377, 421, 481, 483. A script was then used to renumber the sentences.

### Evaluating the model on the gold standard

We now parse corresponding system outputs and get evaluation metrics against the gold standard.

`eval_data/sample_disambig/sample_492.tsv` contains the 492 examples corresponding to the gold standard.

In [1]:
from eval_modules.run_eval import write_sys_from_tsv
from pathlib import Path
from grammar_modules.disambiguation import REPO_ROOT


AMBIG_FILE = REPO_ROOT / "evaluation" / "eval_data" / "sample_disambig" / "sample_492.tsv"
DISAMBIG_OUT = REPO_ROOT / "evaluation" / "eval_data" / "out" / "sys_492_out_v2.txt"
CG3 = REPO_ROOT / "data" / "grammars" / "disambiguation.cg3"
FST = REPO_ROOT / "data" / "fst" / "ojibwe.fomabin"

# System outputs on the 241 examples
write_sys_from_tsv(AMBIG_FILE, DISAMBIG_OUT, CG3, FST)

FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe.fomabin
Wrote 492 sentences to /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/evaluation/eval_data/out/sys_492_out_v2.txt


In [1]:
from eval_modules.run_eval import write_sys_from_tsv
from pathlib import Path
from grammar_modules.disambiguation import REPO_ROOT


AMBIG_FILE = REPO_ROOT / "evaluation" / "eval_data" / "sample_disambig" / "sample_2000.tsv"
DISAMBIG_OUT = REPO_ROOT / "evaluation" / "eval_data" / "out" / "sys_opd_2000.txt"
CG3 = REPO_ROOT / "data" / "grammars" / "disambiguation.cg3"
FST = REPO_ROOT / "data" / "fst" / "ojibwe20260731.fomabin"

# System outputs on the 241 examples
write_sys_from_tsv(AMBIG_FILE, DISAMBIG_OUT, CG3, FST)

FST file is /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe20260731.fomabin
Wrote 2000 sentences to /Users/matthias/labs/ELF-Lab/ELF-Lab Repos/Ojibwe_Constraint_Grammar/evaluation/eval_data/out/sys_opd_2000.txt


#### Get general disambiguation stats on the sample of 492 examples

In [3]:
# get the 492 sentences into a per-line .txt file (disambiguate_with_stats expects this)
from pathlib import Path
import csv

       
OUT = REPO_ROOT / "evaluation" / "eval_data" / "sample_disambig" / "per_line_sample_492.txt"

with AMBIG_FILE.open("r", encoding="utf-8") as fin, OUT.open("w", encoding="utf-8", newline="") as fout:
    reader = csv.DictReader(fin, delimiter="\t")
    for row in reader:
        text = (row.get("Ojibwe") or "").strip()
        if text:
            fout.write(text + "\n")


# Run disambiguation stats on it
from helper_modules.stats_disambiguation import disambiguate_with_stats
from grammar_modules.fst import load_fst_parser

after_block, stats = disambiguate_with_stats(
    text_path=OUT,
    grammar=CG3,
    fst=load_fst_parser(),
    verbose=True, 
)

FST file is /Users/matthias/labs/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin
[############################] 4/4 All done!g text with cg3lmost all running time is spent here!)
|-------------------|----------|
| total words       | 1906     |
| readings before   | 3369     |
| readings after    | 2897     |
| analyses removed  |  472     |
| ambiguity removed |    0.41  |
| ambiguous before  |    0.284 |
| ambiguous after   |    0.167 |

| type    |   words b |   words a |   readings b |   readings a |   removed |   avg b |   avg a |
|---------|-----------|-----------|--------------|--------------|-----------|---------|---------|
| verb    |       807 |       799 |         1442 |         1118 |       324 |    1.79 |    1.4  |
| pronoun |       200 |       200 |          263 |          226 |        37 |    1.31 |    1.13 |
| noun    |       377 |       369 |          503 |          425 |        78 |    1.33 |    1.15 |
| adverb  |       401 |       398 |          404 

#### Compare ambiguity cases to gold standard

In [2]:
from pathlib import Path
from eval_modules.auto_contrast_eval import per_contrast_eval_from_paths

AMBIG_FILE = REPO_ROOT / "evaluation" / "eval_data" / "sample_disambig" / "sample_492.txt"
GOLD_FILE = REPO_ROOT / "evaluation" / "eval_data" / "gold" / "disambig_gold_492.txt"
SYS_FILE = REPO_ROOT / "evaluation" / "eval_data" / "out" / "sys_492_out_v2.txt"

df = per_contrast_eval_from_paths(AMBIG_FILE, GOLD_FILE, SYS_FILE)
df.head(20)


,Contrast,n,Exact,Fail,Exact%,ContainsGold%,Avg sys kept,Examples
0,"OBJ{0PlObj, 0SgObj}",54,36,18,66.666667,100.000000,1.351852,"[{'sent_id': '13', 'token_idx': 1, 'surface': ..."
1,"FORM{ChCnj|Cnj|Pos, Pcp|Pos}",50,26,24,52.000000,100.000000,1.480000,"[{'sent_id': '2', 'token_idx': 3, 'surface': '..."
2,"NOUN{ObvPl, ObvSg, Pl}",41,7,34,17.073171,100.000000,1.902439,"[{'sent_id': '9', 'token_idx': 3, 'surface': '..."
3,"OBJ{3PlObvObj, 3SgObvObj}",37,0,37,0.000000,97.297297,2.054054,"[{'sent_id': '9', 'token_idx': 2, 'surface': '..."
4,"NOUN{ObvPl, ObvSg}",30,0,30,0.000000,100.000000,2.000000,"[{'sent_id': '38', 'token_idx': 4, 'surface': ..."
5,"POS{ADVInter, PCInterj}",25,25,0,100.000000,100.000000,1.000000,"[{'sent_id': '20', 'token_idx': 1, 'surface': ..."
6,"FORM{ChCnj|Cnj|Pos, Cnj|Pos}",24,5,19,20.833333,95.833333,2.000000,"[{'sent_id': '35', 'token_idx': 2, 'surface': ..."
7,"SUBJ{0PlSubj, 0SgSubj}",23,17,6,73.913043,100.000000,1.260870,"[{'sent_id': '8', 'token_idx': 2, 'surface': '..."
8,"NOUN{NA, NI}",22,21,1,95.454545,100.000000,1.045455,"[{'sent_id': '1', 'token_idx': 3, 'surface': '..."
9,"PV{daa, ga}",22,0,22,0.000000,100.000000,2.000000,"[{'sent_id': '10', 'token_idx': 2, 'surface': ..."
